# 01 · EDA — 交易数据概览与业务洞察

**目标**：摸清数据全貌 —— 时间趋势、客户地理、品类结构、退货比例 —— 为后续 RFM 分群和复购预测打基础。

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

df = pd.read_parquet('../data/processed/transactions.parquet')
df.shape, df['customer_id'].nunique(), df['invoice_date'].min(), df['invoice_date'].max()

((824293, 10),
 5939,
 Timestamp('2009-12-01 07:45:00'),
 Timestamp('2011-12-09 12:50:00'))

## 1. 数据规模与完整性

In [2]:
print(f'总行数:        {len(df):,}')
print(f'唯一订单:      {df["invoice"].nunique():,}')
print(f'唯一客户:      {df["customer_id"].nunique():,}')
print(f'唯一 SKU:      {df["stockcode"].nunique():,}')
print(f'时间跨度:      {df["invoice_date"].min().date()} ~ {df["invoice_date"].max().date()}')
print(f'退货行占比:    {df["is_return"].mean():.2%}')
print(f'退货总金额:    £{-df.loc[df["is_return"], "amount"].sum():,.0f}')
print(f'净成交 GMV:    £{df["amount"].sum():,.0f}')

总行数:        824,293
唯一订单:      44,870
唯一客户:      5,939
唯一 SKU:      4,646
时间跨度:      2009-12-01 ~ 2011-12-09
退货行占比:    2.27%
退货总金额:    £1,095,137
净成交 GMV:    £16,648,292


## 2. 月度 GMV 趋势

In [3]:
monthly = (df[~df['is_return']]
           .assign(month=lambda d: d['invoice_date'].dt.to_period('M').dt.to_timestamp())
           .groupby('month', as_index=False)
           .agg(gmv=('amount', 'sum'), orders=('invoice', 'nunique'), customers=('customer_id', 'nunique')))
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(x=monthly['month'], y=monthly['gmv'], name='GMV (£)', marker_color='#4C78A8'), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly['month'], y=monthly['customers'], name='活跃客户数', line=dict(color='#F58518', width=3)), secondary_y=True)
fig.update_layout(title='月度 GMV 与活跃客户数', height=400, hovermode='x unified')
fig.update_yaxes(title_text='GMV (£)', secondary_y=False)
fig.update_yaxes(title_text='活跃客户数', secondary_y=True)
fig.show()
monthly.tail(6)

,month,gmv,orders,customers
19,2011-07-01,600091.011,1331,949
20,2011-08-01,645343.900,1280,935
21,2011-09-01,952838.382,1755,1266
22,2011-10-01,1039318.790,1929,1364
23,2011-11-01,1161817.380,2657,1664
24,2011-12-01,518210.790,778,615


> **洞察**：GMV 呈明显季节性，Q4 冲高（礼品零售的圣诞旺季）；2011-11 达峰值，12 月数据不完整。

## 3. 客户地理分布

In [4]:
country_stats = (df[~df['is_return']]
                 .groupby('country')
                 .agg(customers=('customer_id', 'nunique'),
                      gmv=('amount', 'sum'),
                      orders=('invoice', 'nunique'))
                 .sort_values('gmv', ascending=False)
                 .head(10))
country_stats['gmv_share'] = country_stats['gmv'] / df.loc[~df['is_return'], 'amount'].sum()
country_stats.style.format({'gmv': '£{:,.0f}', 'gmv_share': '{:.1%}'})

,customers,gmv,orders,gmv_share
country,,,,
United Kingdom,5350,"£14,723,148",33541,83.0%
EIRE,5,"£621,631",567,3.5%
Netherlands,22,"£554,232",228,3.1%
Germany,107,"£431,262",789,2.4%
France,95,"£355,257",614,2.0%
Australia,15,"£169,968",95,1.0%
Spain,41,"£109,179",154,0.6%
Switzerland,22,"£100,365",90,0.6%
Sweden,19,"£91,550",104,0.5%


> **洞察**：英国本土贡献 ~80%+ GMV，是典型的本土主导型 B2B 批发商。

## 4. 品类 / 商品集中度 — 长尾结构

In [5]:
prod = (df[~df['is_return']]
        .groupby('stockcode')
        .agg(gmv=('amount', 'sum'), qty=('quantity', 'sum'))
        .sort_values('gmv', ascending=False))
prod['cum_share'] = prod['gmv'].cumsum() / prod['gmv'].sum()
top20_share = prod.head(int(len(prod)*0.2))['gmv'].sum() / prod['gmv'].sum()
print(f'商品总数:               {len(prod):,}')
print(f'TOP 20% SKU 贡献 GMV:   {top20_share:.1%}  ← 帕累托检验')
print(f'TOP 100 SKU 贡献 GMV:   {prod.head(100)["gmv"].sum() / prod["gmv"].sum():.1%}')

商品总数:               4,631
TOP 20% SKU 贡献 GMV:   78.6%  ← 帕累托检验
TOP 100 SKU 贡献 GMV:   30.5%


## 5. 订单金额分布 — 单均与客单

In [6]:
order_totals = df[~df['is_return']].groupby('invoice')['amount'].sum()
print(f'单均订单金额:   £{order_totals.mean():.2f}  (中位 £{order_totals.median():.2f})')
print(f'单均件数:       {df[~df["is_return"]].groupby("invoice")["quantity"].sum().mean():.1f} 件')
cust_totals = df[~df['is_return']].groupby('customer_id')['amount'].sum()
print(f'客户 LTV 中位:  £{cust_totals.median():.0f}')
print(f'客户 LTV TOP1%: £{cust_totals.quantile(0.99):.0f}  (vs 均值 £{cust_totals.mean():.0f})')
fig = px.histogram(np.log10(order_totals[order_totals > 0]), nbins=60,
                   title='订单金额分布（log10）', labels={'value': 'log10(订单金额)'})
fig.update_layout(height=350, showlegend=False)
fig.show()

单均订单金额:   £479.95  (中位 £305.25)
单均件数:       289.6 件


客户 LTV 中位:  £899
客户 LTV TOP1%: £29730  (vs 均值 £3019)


## 6. 退货分析

In [7]:
ret = df[df['is_return']]
print(f'退货订单数:    {ret["invoice"].nunique():,}')
print(f'涉及客户数:    {ret["customer_id"].nunique():,}  ({ret["customer_id"].nunique() / df["customer_id"].nunique():.1%} 客户发生过退货)')
print(f'退货总金额:    £{-ret["amount"].sum():,.0f}  (占 GMV {-ret["amount"].sum() / df[~df["is_return"]]["amount"].sum():.2%})')

退货订单数:    7,901
涉及客户数:    2,572  (43.3% 客户发生过退货)
退货总金额:    £1,095,137  (占 GMV 6.17%)


## 7. 复购行为 — 为后续建模做铺垫

In [8]:
cust_orders = df[~df['is_return']].groupby('customer_id')['invoice'].nunique()
print(f'单次购买客户:   {(cust_orders == 1).sum():,}  ({(cust_orders == 1).mean():.1%})')
print(f'复购客户 (>=2): {(cust_orders >= 2).sum():,}  ({(cust_orders >= 2).mean():.1%})')
print(f'高频客户 (>=10):{(cust_orders >= 10).sum():,}  ({(cust_orders >= 10).mean():.1%})')
fig = px.histogram(cust_orders.clip(upper=30), nbins=30, title='客户订单数分布（截断于 30）',
                   labels={'value': '订单数'})
fig.update_layout(height=350, showlegend=False)
fig.show()

单次购买客户:   1,623  (27.6%)
复购客户 (>=2): 4,255  (72.4%)
高频客户 (>=10):974  (16.6%)


---

## 小结

- 数据整体**干净、体量足够、信号清晰**：82 万行 · 5,939 客户 · 2 年跨度 · 75% 客户有复购
- **季节性 + 长尾 + 高复购率**三个特征组合，非常适合做 RFM 分群与复购预测
- 下一步 → [02_rfm_segmentation.ipynb](02_rfm_segmentation.ipynb)